# Turbulent Flow Training Data Creator

This notebook processes AmiraMesh turbulent flow data into training datasets for SINN models.
It loads velocity field data, computes derived quantities, and prepares training samples.

In [ ]:
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

In [ ]:
def load_amira_lattice_float2(path, shape=(512, 512, 1001), dtype=np.dtype('<f4')):
    """
    Loads an AmiraMesh Lattice { float[2] Data } @1
    Returns: data shaped (nz, ny, nx, 2) by default (time as z).
    
    Parameters:
    -----------
    path : str
        Path to the .am file
    shape : tuple
        (nx, ny, nz) - spatial dimensions and number of timesteps
    dtype : numpy dtype
        Data type (default: little-endian float32)
    
    Returns:
    --------
    data : np.ndarray
        Array of shape (nz, ny, nx, 2) containing velocity components
    """
    nx, ny, nz = shape  # from "define Lattice 512 512 1001"

    with open(path, "rb") as f:
        raw = f.read()

    # Find the '@1' marker, then skip to the start of binary data after the newline
    marker = raw.find(b"@1")
    if marker == -1:
        raise ValueError("Could not find '@1' data marker in file.")

    # Data starts after '@1' and the following newline(s)
    data_start = marker + 2
    while data_start < len(raw) and raw[data_start] in (ord('\n'), ord('\r'), ord(' '), ord('\t')):
        data_start += 1

    # Interpret the remaining bytes as little-endian float32
    arr = np.frombuffer(raw, dtype=dtype, offset=data_start)

    expected = nx * ny * nz * 2
    if arr.size < expected:
        raise ValueError(
            f"File truncated. Got {arr.size}, expected at least {expected}."
        )

    arr = arr[:expected]  # Ignore trailing padding

    # Amira Lattice typically stores x fastest, then y, then z (time)
    data = arr.reshape((nz, ny, nx, 2))
    return data


# -------------------------
# Load data
# -------------------------
print("Loading AmiraMesh data...")
amira_file = "0000.am"  # Change to "0001.am" if desired
data = load_amira_lattice_float2(amira_file, shape=(512, 512, 1001))

print(f"Raw data shape: {data.shape}")  # (1001, 512, 512, 2)
print(f"Data type: {data.dtype}")
print(f"Data range: [{data.min():.4f}, {data.max():.4f}]")
print(f"Memory size: {data.nbytes / 1e6:.1f} MB")

In [ ]:
# -------------------------
# Extract velocity components (time, y, x, 2) -> (time, y, x)
# -------------------------
u = data[:, :, :, 0]  # x-component of velocity
v = data[:, :, :, 1]  # y-component of velocity

print(f"u-component shape: {u.shape}")
print(f"v-component shape: {v.shape}")
print(f"u range: [{u.min():.4f}, {u.max():.4f}]")
print(f"v range: [{v.min():.4f}, {v.max():.4f}]")

# -------------------------
# Compute derived fields
# -------------------------
speed = np.sqrt(u**2 + v**2)  # speed magnitude
print(f"\nSpeed magnitude:")
print(f"  Shape: {speed.shape}")
print(f"  Range: [{speed.min():.4f}, {speed.max():.4f}]")
print(f"  Mean: {speed.mean():.4f}")

# Compute vorticity (simplified 2D: dv/dx - du/dy)
# Using central differences where possible
vorticity = np.zeros_like(speed)
for t in range(speed.shape[0]):
    dv_dx = np.gradient(v[t, :, :], axis=1)
    du_dy = np.gradient(u[t, :, :], axis=0)
    vorticity[t, :, :] = dv_dx - du_dy

print(f"\nVorticity:")
print(f"  Range: [{vorticity.min():.4f}, {vorticity.max():.4f}]")
print(f"  Mean: {vorticity.mean():.4f}")

In [ ]:
# -------------------------
# Store original values for inverse transform
# -------------------------
u_original = u.copy()
v_original = v.copy()
speed_original = speed.copy()

# -------------------------
# Normalize to zero mean, unit variance (per location across time)
# -------------------------
print("Normalizing fields...")

# Normalize u component
u_mean = np.mean(u, axis=0, keepdims=True)
u_std = np.std(u, axis=0, keepdims=True)
u_std = np.maximum(u_std, 1e-8)  # Avoid division by zero
u_norm = (u - u_mean) / u_std

# Normalize v component
v_mean = np.mean(v, axis=0, keepdims=True)
v_std = np.std(v, axis=0, keepdims=True)
v_std = np.maximum(v_std, 1e-8)
v_norm = (v - v_mean) / v_std

# Normalize speed
speed_mean = np.mean(speed, axis=0, keepdims=True)
speed_std = np.std(speed, axis=0, keepdims=True)
speed_std = np.maximum(speed_std, 1e-8)
speed_norm = (speed - speed_mean) / speed_std

print(f"Normalized u: mean={u_norm.mean():.6f}, std={u_norm.std():.6f}")
print(f"Normalized v: mean={v_norm.mean():.6f}, std={v_norm.std():.6f}")
print(f"Normalized speed: mean={speed_norm.mean():.6f}, std={speed_norm.std():.6f}")

# -------------------------
# Check for NaN or Inf values
# -------------------------
print(f"\nData quality check:")
print(f"  NaN values in u_norm: {np.isnan(u_norm).sum()}")
print(f"  NaN values in v_norm: {np.isnan(v_norm).sum()}")
print(f"  Inf values in u_norm: {np.isinf(u_norm).sum()}")
print(f"  Inf values in v_norm: {np.isinf(v_norm).sum()}")

In [ ]:
# -------------------------
# Create spatial grids (normalized to [0, 1])
# -------------------------
nt, ny, nx = u_norm.shape
print(f"Data dimensions: nt={nt}, ny={ny}, nx={nx}")

# Create 2D meshgrid for spatial coordinates
x = np.linspace(0, 1, nx, dtype=np.float32)
y = np.linspace(0, 1, ny, dtype=np.float32)
X, Y = np.meshgrid(x, y)

print(f"X shape: {X.shape}, range: [{X.min():.2f}, {X.max():.2f}]")
print(f"Y shape: {Y.shape}, range: [{Y.min():.2f}, {Y.max():.2f}]")

# -------------------------
# Create time array (index-based)
# -------------------------
T = np.arange(nt, dtype=np.float32)

print(f"T shape: {T.shape}")
print(f"T range: [{T.min():.0f}, {T.max():.0f}]")

# -------------------------
# Convert to XUYT format (expected by solvers)
# U should be a list of 2D arrays (one per timestep)
# -------------------------
print("\nConverting to XUYT format...")

# Option 1: Use normalized speed as the field
U_speed = [speed_norm[t, :, :] for t in range(nt)]
print(f"U_speed: {len(U_speed)} timesteps, each shape {U_speed[0].shape}")

# Option 2: Use normalized velocity magnitude
velocity_magnitude = np.sqrt(u_norm**2 + v_norm**2)
U_velocity = [velocity_magnitude[t, :, :] for t in range(nt)]
print(f"U_velocity: {len(U_velocity)} timesteps, each shape {U_velocity[0].shape}")

# For training, we'll use speed magnitude
U = U_speed

In [ ]:
# -------------------------
# Configuration for training samples
# -------------------------
temporal_window = 10    # Number of timesteps per sample
temporal_stride = 5     # Stride between samples in time
spatial_patch_size = 128  # Size of spatial patches (spatial_patch_size x spatial_patch_size)
spatial_stride = 64     # Stride for spatial patches

print("Generating training samples...")
print(f"  Temporal window: {temporal_window}, stride: {temporal_stride}")
print(f"  Spatial patch: {spatial_patch_size}x{spatial_patch_size}, stride: {spatial_stride}")

# -------------------------
# Generate temporal windows
# -------------------------
temporal_samples = []
for t_start in range(0, nt - temporal_window, temporal_stride):
    t_end = t_start + temporal_window
    temporal_samples.append((t_start, t_end))

print(f"Generated {len(temporal_samples)} temporal windows")

# -------------------------
# Generate spatial patches (for high-res datasets, optional)
# -------------------------
# For now, we'll keep the full spatial domain if it's not too large
if nx > spatial_patch_size or ny > spatial_patch_size:
    spatial_samples = []
    for y_start in range(0, ny - spatial_patch_size, spatial_stride):
        for x_start in range(0, nx - spatial_patch_size, spatial_stride):
            y_end = y_start + spatial_patch_size
            x_end = x_start + spatial_patch_size
            spatial_samples.append((y_start, y_end, x_start, x_end))
    use_patches = True
else:
    spatial_samples = [(0, ny, 0, nx)]
    use_patches = False

print(f"Generated {len(spatial_samples)} spatial patches")
print(f"Use spatial patches: {use_patches}")

# -------------------------
# Create training sample dictionary
# -------------------------
training_samples = []

for t_idx, (t_start, t_end) in enumerate(temporal_samples[:5]):  # Use first 5 temporal windows for now
    for s_idx, (y_start, y_end, x_start, x_end) in enumerate(spatial_samples[:5]):  # Limit to 5 spatial patches
        sample = {
            'temporal_range': (t_start, t_end),
            'spatial_range': (y_start, y_end, x_start, x_end),
            'u': u_norm[t_start:t_end, y_start:y_end, x_start:x_end],
            'v': v_norm[t_start:t_end, y_start:y_end, x_start:x_end],
            'speed': speed_norm[t_start:t_end, y_start:y_end, x_start:x_end],
        }
        training_samples.append(sample)

print(f"\nGenerated {len(training_samples)} training samples")
print(f"Sample size: ({temporal_window}, {y_end-y_start}, {x_end-x_start})")

In [ ]:
# -------------------------
# Visualize full-domain snapshots
# -------------------------
print("Creating visualizations...")

def create_field_visualization(X, Y, snapshots, times, n_indices, vmin, vmax, title_prefix="", save_path=None):
    """Create a grid of contour plots showing field evolution."""
    indices = np.linspace(0, len(snapshots) - 1, min(n_indices, len(snapshots)), dtype=int).tolist()
    ncols = int(np.ceil(len(indices) / 2))

    fig = plt.figure(figsize=(16, 10))
    gs = GridSpec(2, ncols + 1, width_ratios=[1]*ncols + [0.05], figure=fig)

    for i, idx in enumerate(indices):
        row = i // ncols
        col = i % ncols
        ax = fig.add_subplot(gs[row, col])
        
        ax.contourf(X, Y, snapshots[idx], levels=20, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.contour(X, Y, snapshots[idx], levels=8, colors='black', linewidths=0.5, alpha=0.3)
        ax.set_title(f'{title_prefix} at t={idx}')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_aspect('equal')

    cax = fig.add_subplot(gs[:, -1])
    sm = ScalarMappable(cmap='viridis', norm=Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    fig.colorbar(sm, cax=cax)
    cax.set_ylabel('Magnitude', rotation=270, labelpad=15)

    fig.suptitle(f'{title_prefix} Evolution (Normalized)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved to {save_path}")
    
    plt.show()
    return fig

# Visualize speed magnitude
vmin_speed = np.min([np.min(u) for u in U])
vmax_speed = np.max([np.max(u) for u in U])
fig1 = create_field_visualization(X, Y, U, T, n_indices=12, vmin=vmin_speed, vmax=vmax_speed,
                                   title_prefix="Speed Magnitude", save_path="speed_visualization.png")

# -------------------------
# Plot statistics
# -------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Speed statistics over time
speed_min_over_time = [np.min(U[t]) for t in range(len(U))]
speed_max_over_time = [np.max(U[t]) for t in range(len(U))]
speed_mean_over_time = [np.mean(U[t]) for t in range(len(U))]

axes[0, 0].plot(T, speed_min_over_time, label='Min', linewidth=2)
axes[0, 0].plot(T, speed_mean_over_time, label='Mean', linewidth=2)
axes[0, 0].plot(T, speed_max_over_time, label='Max', linewidth=2)
axes[0, 0].set_xlabel('Time index')
axes[0, 0].set_ylabel('Speed (normalized)')
axes[0, 0].set_title('Speed Statistics Over Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Spatial distribution of mean speed
spatial_mean = np.mean(speed_norm, axis=0)
im2 = axes[0, 1].imshow(spatial_mean, cmap='viridis', origin='lower')
axes[0, 1].set_title('Mean Speed (spatial)')
axes[0, 1].set_xlabel('X index')
axes[0, 1].set_ylabel('Y index')
plt.colorbar(im2, ax=axes[0, 1])

# Plot 3: Spatial distribution of speed variance
spatial_var = np.var(speed_norm, axis=0)
im3 = axes[1, 0].imshow(spatial_var, cmap='hot', origin='lower')
axes[1, 0].set_title('Speed Variance (spatial)')
axes[1, 0].set_xlabel('X index')
axes[1, 0].set_ylabel('Y index')
plt.colorbar(im3, ax=axes[1, 0])

# Plot 4: Histogram of speed values
all_speeds = np.concatenate([U[t].flatten() for t in range(len(U))])
axes[1, 1].hist(all_speeds, bins=100, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Speed (normalized)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Speed Distribution (all time and space)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data_statistics.png', dpi=150, bbox_inches='tight')
print("✓ Saved statistics figure to data_statistics.png")
plt.show()

In [ ]:
# -------------------------
# Prepare complete dataset for SINN training
# -------------------------
print("\nPreparing complete dataset...")

# Create comprehensive data dictionary
training_data = {
    # Core XUYT format for SINN solver
    'X': X,                           # 2D spatial grid
    'Y': Y,                           # 2D spatial grid
    'U': U,                           # List of 2D arrays (speed magnitude)
    'T': T,                           # Time indices
    
    # Original (non-normalized) data
    'U_original': U_original,         # Original speed magnitude (non-normalized)
    'u_original': u_original,         # Original u component
    'v_original': v_original,         # Original v component
    
    # Normalized data (for direct use or reference)
    'u_normalized': u_norm,           # Normalized u component (3D array)
    'v_normalized': v_norm,           # Normalized v component (3D array)
    'speed_normalized': speed_norm,   # Normalized speed magnitude (3D array)
    'vorticity': vorticity,           # Vorticity field
    
    # Normalization parameters (for inverse transform)
    'normalization': {
        'u_mean': u_mean,
        'u_std': u_std,
        'v_mean': v_mean,
        'v_std': v_std,
        'speed_mean': speed_mean,
        'speed_std': speed_std,
    },
    
    # Training samples (optional, for convenience)
    'training_samples': training_samples,
    
    # Metadata
    'metadata': {
        'source_file': amira_file,
        'description': 'Turbulent flow dataset from AmiraMesh file',
        'shape': {
            'time': nt,
            'spatial_y': ny,
            'spatial_x': nx,
        },
        'normalization_method': 'per-location z-score (zero mean, unit variance)',
        'timestamp': np.datetime_as_string(np.datetime64('now')),
    }
}

print("Dataset structure:")
for key in training_data.keys():
    if key != 'training_samples':
        if isinstance(training_data[key], np.ndarray):
            print(f"  {key}: {training_data[key].shape} {training_data[key].dtype}")
        elif isinstance(training_data[key], list):
            print(f"  {key}: list of {len(training_data[key])} items")
        elif isinstance(training_data[key], dict):
            print(f"  {key}: dict with keys {list(training_data[key].keys())}")
    else:
        print(f"  {key}: {len(training_data[key])} samples")

# -------------------------
# Save to pickle file
# -------------------------
output_filename = "training_data_turbulent_flow.pkl"
print(f"\nSaving training data to {output_filename}...")

with open(output_filename, "wb") as f:
    pickle.dump(training_data, f)

file_size_mb = os.path.getsize(output_filename) / 1e6
print(f"✓ Saved successfully!")
print(f"  File size: {file_size_mb:.1f} MB")

# -------------------------
# Load and verify
# -------------------------
print("\nVerifying saved data...")
with open(output_filename, "rb") as f:
    loaded_data = pickle.load(f)

print(f"✓ Verification successful!")
print(f"  X shape: {loaded_data['X'].shape}")
print(f"  Y shape: {loaded_data['Y'].shape}")
print(f"  U length: {len(loaded_data['U'])}")
print(f"  T shape: {loaded_data['T'].shape}")

print("\n" + "="*60)
print("TRAINING DATA CREATION COMPLETE")
print("="*60)
print(f"\nNext steps to use this data:")
print(f"1. Load the data:")
print(f"   with open('{output_filename}', 'rb') as f:")
print(f"       data = pickle.load(f)")
print(f"\n2. Create SINN solver:")
print(f"   from sinn_solver import sinn")
print(f"   solver = sinn(data['X'], data['Y'], data['U'], data['T'])")
print(f"\n3. Train the model:")
print(f"   solver.train(num_epochs=1000, batch_size=32)")

## 8. Save Training Dataset to File

Save the processed training data to a pickle file with organized structure and metadata.


## 7. Visualize Sample Training Data

Create visualizations showing the training data snapshots and field statistics.


## 6. Generate Training Samples with Sliding Windows

Create training samples using sliding windows over time and space.


## 5. Create Spatial and Temporal Grids

Create coordinate grids for spatial dimensions and time indices in XUYT format.


## 4. Normalize and Preprocess Data

Normalize velocity components and derived fields to standardized ranges.


## 3. Extract Velocity Components and Compute Derived Fields

Extract u and v velocity components from the loaded data and compute derived quantities.


## 2. Load Amira Lattice Data

Define function to load AmiraMesh files and verify data integrity.


## 1. Import Required Libraries

Import necessary libraries for data manipulation, visualization, and persistence.
